# QSVT segment-level study — solution amplitudes and angles

Mirror of `02_solution_amplitudes_and_angles.ipynb` with the **QSVT** solver added:
per-segment solution-amplitude histograms (true vs false) for classical, 1BQF and
QSVT, and the per-segment polar-angle (θ from beam axis) distributions of the
*active* segments. Non-classical solutions are rescaled to the classical signal
support (`seg_store.load_vectors`, same convention as the metrics view).
γ = 3, clean events.

In [1]:
import sys
sys.path.insert(0, "/data/bfys/gscriven/Quantum_Track_Reconstruction/Toy_Characterisation/Segment_level_studies")
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seg_store as S
plt.rcParams.update({"figure.dpi":110,"font.size":11,"axes.grid":True,"grid.alpha":0.3})
OUT=Path("/data/bfys/gscriven/Quantum_Track_Reconstruction/QSVT/Segment_level_studies/outputs/amplitudes_and_angles")
OUT.mkdir(parents=True,exist_ok=True)
GAMMA=3.0; TAU=S.threshold(GAMMA)
idxC=S.solves_index("classical",GAMMA,0.0)
def grab(solver,n_trk,rep=0):
    idx=S.solves_index(solver,GAMMA,0.0)
    r=idx[(idx.n_trk==n_trk)&(idx.rep==rep)]
    if not len(r): return None
    return S.load_vectors(r.iloc[0],classical_partner=idxC)

## 1. Amplitude histograms — true vs false, three solvers (T = 100 and 400)

In [2]:
for T in (100,400):
    fig,ax=plt.subplots(1,3,figsize=(16,4.8),sharey=True)
    for a,(slv,name) in zip(ax,[("classical","classical 1/λ"),("quantum","1BQF (cos)"),("qsvt","QSVT (deg 40)")]):
        v=grab(slv,T)
        if v is None: a.set_title(f"{name}: no solve"); continue
        s,t=np.abs(v["sol"]),v["truth"]
        bins=np.linspace(0,max(0.7,np.percentile(s,99.9)),80)
        a.hist(s[~t],bins=bins,color="#d6604d",alpha=0.6,log=True,label=f"false (n={int((~t).sum())})")
        a.hist(s[t],bins=bins,color="#1b7837",alpha=0.7,log=True,label=f"true (n={int(t.sum())})")
        a.axvline(TAU,color="k",ls=":",lw=1.5,label=f"τ={TAU}")
        a.set_xlabel("solution amplitude (signal-rescaled)"); a.set_title(name,fontweight="bold"); a.legend(fontsize=8)
    ax[0].set_ylabel("# segments (log)")
    fig.suptitle(f"Solution amplitudes, true vs false — γ=3, T={T}, clean",fontsize=12.5,fontweight="bold",y=1.0)
    fig.tight_layout(rect=[0,0,1,0.95])
    for ext,dpi in (("pdf",600),("png",300)): fig.savefig(OUT/f"qsvt_amplitudes_T{T}.{ext}",dpi=dpi,bbox_inches="tight",facecolor="white")
    plt.show(); print(f"saved qsvt_amplitudes_T{T}")

saved qsvt_amplitudes_T100


saved qsvt_amplitudes_T400


## 2. Angle of the *active* segments — does QSVT keep the activated set collimated?

True segments are collimated near the beam axis; cross-track false segments spread
to larger θ. A solver that activates false segments pollutes the large-θ tail.

In [3]:
T=400
fig,ax=plt.subplots(1,3,figsize=(16,4.8),sharey=True)
for a,(slv,name) in zip(ax,[("classical","classical 1/λ"),("quantum","1BQF (cos)"),("qsvt","QSVT (deg 40)")]):
    v=grab(slv,T)
    if v is None: a.set_title(f"{name}: no solve"); continue
    s,t,th=np.abs(v["sol"]),v["truth"],v["theta_mrad"]
    act=s>TAU
    bins=np.linspace(0,min(260,np.percentile(th,99.5)),60)
    a.hist(th[act&~t],bins=bins,color="#d6604d",alpha=0.65,log=True,label=f"active false ({int((act&~t).sum())})")
    a.hist(th[act&t],bins=bins,color="#1b7837",alpha=0.7,log=True,label=f"active true ({int((act&t).sum())})")
    a.set_xlabel("segment polar angle θ (mrad)"); a.set_title(name,fontweight="bold"); a.legend(fontsize=8)
ax[0].set_ylabel("# active segments (log)")
fig.suptitle(f"Active-segment angles — γ=3, T={T}, clean (τ={TAU})",fontsize=12.5,fontweight="bold",y=1.0)
fig.tight_layout(rect=[0,0,1,0.95])
for ext,dpi in (("pdf",600),("png",300)): fig.savefig(OUT/f"qsvt_angles_T{T}.{ext}",dpi=dpi,bbox_inches="tight",facecolor="white")
plt.show(); print("saved qsvt_angles_T400")

saved qsvt_angles_T400


In [4]:
# summary: false-active counts and the amplitude attractors per solver at T=400
rows=[]
for slv in ("classical","quantum","qsvt"):
    v=grab(slv,400)
    if v is None: continue
    s,t=np.abs(v["sol"]),v["truth"]
    rows.append(dict(solver=slv,n_false_active=int(((s>TAU)&~t).sum()),
                     med_true=round(float(np.median(s[t])),3),
                     med_false=round(float(np.median(s[~t])),4),
                     p99_false=round(float(np.percentile(s[~t],99)),3),
                     min_true=round(float(s[t].min()),3)))
print(pd.DataFrame(rows).to_string(index=False))

   solver  n_false_active  med_true  med_false  p99_false  min_true
classical              33     0.455      0.250      0.250     0.364
  quantum              21     0.443      0.000      0.000     0.170
     qsvt               0     0.377      0.002      0.002     0.006


## 3. Read-off

- **Classical:** false grass at the 0.25 attractor with a heavy tail past τ at
  T = 400 (the known false-positive growth).
- **1BQF:** isolated grass erased (on the notch), but the true band is split by
  the cosine weighting (outer segments pushed toward τ) and the surviving
  coupled false stay.
- **QSVT (line comb):** the grass is pushed orders of magnitude below the signal
  *and* the true band stays tight near the classical values; the active set
  remains collimated, with false activations at or near zero even at T = 400.